In [1]:
%pip install --quiet --upgrade transformers datasets sagemaker s3fs

%pip uninstall -y sagemaker-core sagemaker-train sagemaker-serve sagemaker-mlops

%pip install --quiet --no-cache-dir --force-reinstall "sagemaker>=2.245.0,<3"

# sagemaker introduced a latest version 3 but it doesn't have many features that supports hugging face,datasets etc. so the sagemaker needs to something greater than 2.245 and 3


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-multimodal 1.5.0 requires nvidia-ml-py3<8.0,>=7.352.0, which is not installed.
autogluon-timeseries 1.5.0 requires chronos-forecasting<2.4,>=2.2.2, which is not installed.
autogluon-timeseries 1.5.0 requires einops<1,>=0.7, which is not installed.
autogluon-timeseries 1.5.0 requires peft<0.18,>=0.13.0, which is not installed.
amazon-sagemaker-sql-magic 0.1.4 requires numpy<2, but you have numpy 2.5.2 which is incompatible.
autogluon-common 1.5.0 requires numpy<2.4.0,>=1.25.0, but you have numpy 2.5.2 which is incompatible.
autogluon-common 1.5.0 requires psutil<7.2.0,>=5.7.3, but you have psutil 7.2.2 which is incompatible.
autogluon-common 1.5.0 requires pyarrow<21.0.0,>=7.0.0, but you have pyarrow 25.0.1 which is incompatible.
autogluon-core 1.5.0 requires numpy<2.4.0,>=1.25.0, but you have numpy 2.5.2

In [2]:
import sagemaker
import boto3 # python sdk for AWS

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


/opt/conda/lib/python3.12/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


In [3]:
sess=sagemaker.Session() # it gets session of the the sagemaker notebook
sess

In [4]:
role=sagemaker.get_execution_role() # Role of the sagemaker notebook basically an IAM role for the notebook
role 

'<SAGEMAKER_EXECUTION_ROLE_ARN>'

In [5]:
sess.boto_region_name # notebook session, this session must be the same as in s3 location if you are using training data from s3

'us-east-2'

In [ ]:
from datasets import load_dataset
from random import randrange

dataset = load_dataset("databricks/databricks-dolly-15k",split="train") #prompt response
print(f"dataset size:{len(dataset)}")
print(dataset[randrange(len(dataset))]) # select a random item from entire dataset

# https://huggingface.co/datasets/databricks/databricks-dolly-15k
# the more diverse the dataset is the better the model is at answering at the specific way we desire it. 


In [ ]:
def format_dolly(sample):
    instruction = f"### Instruction\n{sample['instruction']}"
    context=f"### Context\n{sample['context']}" if len(sample['context'])>0 else None
    response=f"### Answer\n{sample['response']}"
    prompt="\n\n".join(i for i in [instruction,context,response] if i is not None)
    return prompt
#  when we fine-tune these llms there is a specific way how we should format our instructions

In [ ]:
print(format_dolly(dataset[randrange(len(dataset))]))

In [ ]:
import os

os.environ["HF_FINE_TUNING_TOKEN"]="<HUGGING_FACE_TOKEN>"


In [6]:
from transformers import AutoTokenizer

model_id = 'mistralai/Mixtral-8x7B-v0.1'
tokenizer = AutoTokenizer.from_pretrained(model_id) # Tokenization
tokenizer.pad_token=tokenizer.eos_token # this line sets the padding token to be same as end of sequence (eos) token. This typically done when model doesn't have the predefined padding token enabling the batching of sequences with different length by using the eos token to fill the shorter sequences. Reason is some prompt are short and other are longer so use these to maintain the same length inputs that will be given to to the model   


In [ ]:
dataset.features

In [ ]:
from random import randint
from itertools import chain
from functools import partial

def template_dataset(sample):
    sample['text']= f"{format_dolly(sample)}{tokenizer.eos_token}" # adding end of sequence so the model knows thats the end
    return sample

dataset = dataset.map(template_dataset,remove_columns=list(dataset.features)) # apply template_dataset function to every sample in the dataset quickly and efficiently 
# above remove [instruction, context, response, category] we will have a sample i.e text field which encapsulate all of this 
print(dataset[randint(0,len(dataset))]['text'])

remainder = {'input_ids':[],'attention_mask':[],'token_type_ids':[]}
# We don't pad individual examples here because we use sequence packing.
# Multiple tokenized examples are concatenated and split into fixed
# 2048-token blocks.

# Instead of doing the following 
#Sample A: [700 real tokens]  + [1348 PAD]
#Sample B: [900 real tokens]  + [1148 PAD]
#Sample C: [600 real tokens]  + [1448 PAD]

# the below is done
# you concatenate them Sample A + Sample B + Sample C

#Block 1 = 2048 real tokens
#Block 2 = 2048 real tokens
#Block 3 = 2048 real tokens

# Therefore, complete blocks contain only real tokens, so their
# attention_mask is 2048 ones:
# [1, 1, 1, ..., 1]

# EOS separates individual training examples inside the packed stream.
# It is NOT serving as padding here.

# remainder stores tokens that are not enough to form another
# complete 2048-token block. They can be carried into the next batch.

def chunk(sample, chunk_length = 2048):
    
    global remainder
    #print(len(sample["input_ids"]),"first")
    #print(len(sample["input_ids"][999]),"second")

    concatenated_examples = {k: list(chain(*sample[k])) for k in sample.keys()}

    concatenated_examples = {
          k: remainder[k]+concatenated_examples[k] for k in concatenated_examples.keys()
    }

    batch_total_length = len(concatenated_examples[list(sample.keys())[0]])
    #print(batch_total_length,"third")
    
    if batch_total_length >= chunk_length:
        batch_total_length = (batch_total_length // chunk_length) * chunk_length

    #print(batch_total_length,"fourth")
    result = {
        k: [t[i:i+chunk_length] for i in range(0, batch_total_length, chunk_length)]  for k,t in concatenated_examples.items()
    }

    remainder = {
       k: concatenated_examples[k][batch_total_length:] for k in concatenated_examples.keys()
    }
    
    #print(len(result['input_ids']),"fifth")
    result["labels"] = result["input_ids"].copy()
    #raise Exception("Stop")
    return result

lm_dataset = dataset.map(
    lambda sample: tokenizer(sample["text"]),
    batched= True, # With batched=True, sample["text"] is actually a list of texts, something like
    remove_columns = list(dataset.features)
).map(
   partial(chunk,chunk_length=2048), # apply the chunk() function to each batch, while permanently supplying chunk_length as an extra argument. thats the reason map is used too.
    batched= True
)
print(f"Total number of samples: {len(lm_dataset)}") # you have this many chunks, and each is 2048 tokens long

In [ ]:
import datasets
import transformers

print("datasets version:", datasets.__version__)
print("transformers version:", transformers.__version__)

print(lm_dataset)
print(lm_dataset.features)

In [ ]:
#<DATASET_BUCKET>
import s3fs

training_input_path = f"s3://<DATASET_BUCKET>/processed/mixtral/dolly/train"

lm_dataset.save_to_disk(training_input_path)
print("uploading the dataset to s3")



In [7]:
training_input_path = f"s3://<DATASET_BUCKET>/processed/mixtral/dolly/train"


In [8]:
import time
from sagemaker.huggingface import HuggingFace


job_name = f"mixtral-8x7b-qlora-{time.strftime('%Y-%m-%d-%H-%M-%S', time.localtime())}"

hyperparameters = {
    "model_id": model_id,
    "dataset_path": "/opt/ml/input/data/training", # local path from s3 it download it to local path
    "epochs": 2,
    "per_device_train_batch_size": 2, # batch size per Gpu i.e 2 samples for one GPU  if we have 4 GPUs then 8 samples 
    "lr": 2e-4,
    "merge_weights": True,
}

huggingface_estimator = HuggingFace(
    entry_point = "run_clm.py",
    source_dir= "scripts",
    instance_type = "ml.g5.24xlarge",
    instance_count = 1,
    base_job_name = job_name,
    role = role,
    volume_size = 300,
    transformers_version= "4.28",
    pytorch_version= "2.0",
    py_version= "py310",
    hyperparameters = hyperparameters,
    environment = {
        "HUGGINGFACE_HUB_CACHE": "/tmp/.cache"
    },
    disable_profiler=True,
    debugger_hook_config=False,
)

/opt/conda/lib/python3.12/site-packages/sagemaker/estimator.py:588: SageMakerV2DeprecationWarning: HuggingFace is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelTrainer` (`from sagemaker.train import ModelTrainer`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
HuggingFace is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelTrainer` (`from sagemaker.train import ModelTrainer`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


In [9]:
data = {"training": training_input_path}
huggingface_estimator.fit(data,wait = True)

INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: mixtral-8x7b-qlora-2026-08-12-21-08-11-2026-08-12-21-08-11-975


2026-08-12 21:08:13 Starting - Starting the training job...
2026-08-12 21:08:37 Starting - Preparing the instances for training...
2026-08-12 21:09:13 Downloading - Downloading input data...
2026-08-12 21:09:23 Downloading - Downloading the training image..................
2026-08-12 21:12:30 Training - Training image download completed. Training in progress.....Downloading shards:  26%|██▋       | 5/19 [00:50<02:20, 10.00s/it]
Loading checkpoint shards: 100%|██████████| 19/19 [00:30<00:00,  1.61s/it]
found 8 modules to quantize: ['w1', 'o_proj', 'w2', 'gate', 'k_proj', 'w3', 'q_proj', 'v_proj']
trainable params: 968,900,608 || all params: 24,451,502,080 || trainable%: 3.96254023507418
4%|▍         | 60/1528 [15:08<6:09:53, 15.12s/it]
